# Change Blindness

> **Task name:** `Change Blindness`

**Track: Attention** | Can the model detect subtle changes across an interruption?

Humans often fail to notice changes in a scene when the change coincides with a
brief disruption (a blink, a flicker, a cut). This benchmark tests whether LLMs
can detect factual changes between two versions of a passage when separated by
a disruptor paragraph.

**Metrics:**
- Detection rate by change magnitude (minor vs. major)
- Detection rate by disruptor length (0, 1, 3 paragraphs)
- False alarm rate (reporting non-existent changes)
- Specificity (identifying exactly what changed)

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.lower().strip())

In [ ]:
# --- Passages with controlled changes ---
# Each passage has a "minor" change (a number/name) and a "major" change (a logical claim).
# Passages are about coding and software engineering — natural territory for LLMs.

PASSAGES = [
    {
        "id": "api",
        "original": "The payments API accepts POST requests to /v2/charges with a JSON body containing amount, currency, and source fields. Rate limiting is set to 500 requests per minute per API key. All responses include an idempotency_key header to prevent duplicate charges. Failed requests return a 402 status code with an error object containing a decline_code field.",
        "minor_change": "The payments API accepts POST requests to /v2/charges with a JSON body containing amount, currency, and source fields. Rate limiting is set to 200 requests per minute per API key. All responses include an idempotency_key header to prevent duplicate charges. Failed requests return a 402 status code with an error object containing a decline_code field.",
        "minor_detail": "500 requests per minute changed to 200 requests per minute",
        "major_change": "The payments API accepts POST requests to /v2/charges with a JSON body containing amount, currency, and source fields. Rate limiting is set to 500 requests per minute per API key. All responses include an idempotency_key header to prevent duplicate charges. Failed requests return a 200 status code with the charge marked as successful regardless of the actual outcome.",
        "major_detail": "402 status with decline_code changed to 200 status marked as successful regardless of outcome",
    },
    {
        "id": "migration",
        "original": "The database migration adds a composite index on (user_id, created_at) to the orders table, which currently holds 12 million rows. The migration script runs in batches of 10,000 rows to avoid locking the table. Estimated completion time is 45 minutes during the maintenance window. A rollback script drops the index and is tested to complete in under 3 minutes.",
        "minor_change": "The database migration adds a composite index on (user_id, created_at) to the orders table, which currently holds 12 million rows. The migration script runs in batches of 50,000 rows to avoid locking the table. Estimated completion time is 45 minutes during the maintenance window. A rollback script drops the index and is tested to complete in under 3 minutes.",
        "minor_detail": "batches of 10,000 rows changed to batches of 50,000 rows",
        "major_change": "The database migration adds a composite index on (user_id, created_at) to the orders table, which currently holds 12 million rows. The migration script runs in batches of 10,000 rows to avoid locking the table. Estimated completion time is 45 minutes during the maintenance window. There is no rollback script because the migration is irreversible once started.",
        "major_detail": "rollback script drops index in under 3 minutes changed to no rollback script because migration is irreversible",
    },
    {
        "id": "deploy",
        "original": "The CI pipeline runs 3,200 unit tests in parallel across 8 workers, completing in about 4 minutes. Coverage must stay above 85 percent or the build fails. After tests pass, the Docker image is pushed to the registry tagged with the commit SHA. Deployments to production use a blue-green strategy with automatic rollback if the health check fails within 60 seconds.",
        "minor_change": "The CI pipeline runs 3,200 unit tests in parallel across 4 workers, completing in about 4 minutes. Coverage must stay above 85 percent or the build fails. After tests pass, the Docker image is pushed to the registry tagged with the commit SHA. Deployments to production use a blue-green strategy with automatic rollback if the health check fails within 60 seconds.",
        "minor_detail": "8 workers changed to 4 workers",
        "major_change": "The CI pipeline runs 3,200 unit tests in parallel across 8 workers, completing in about 4 minutes. Coverage must stay above 85 percent or the build fails. After tests pass, the Docker image is pushed to the registry tagged with the commit SHA. Deployments to production go straight to all servers at once with no rollback mechanism if something breaks.",
        "major_detail": "blue-green strategy with automatic rollback changed to all servers at once with no rollback mechanism",
    },
    {
        "id": "cache",
        "original": "The Redis cache layer sits between the application servers and PostgreSQL, storing serialized JSON with a TTL of 15 minutes. Cache keys follow the pattern user:{id}:profile and are invalidated on write. The cache hit rate averages 92 percent during peak traffic. When Redis is unavailable, the app falls back to direct database queries with a circuit breaker that trips after 5 consecutive failures.",
        "minor_change": "The Redis cache layer sits between the application servers and PostgreSQL, storing serialized JSON with a TTL of 15 minutes. Cache keys follow the pattern user:{id}:profile and are invalidated on write. The cache hit rate averages 78 percent during peak traffic. When Redis is unavailable, the app falls back to direct database queries with a circuit breaker that trips after 5 consecutive failures.",
        "minor_detail": "92 percent hit rate changed to 78 percent hit rate",
        "major_change": "The Redis cache layer sits between the application servers and PostgreSQL, storing serialized JSON with a TTL of 15 minutes. Cache keys follow the pattern user:{id}:profile and are invalidated on write. The cache hit rate averages 92 percent during peak traffic. When Redis is unavailable, the app returns stale cached data indefinitely and never queries the database directly.",
        "major_detail": "falls back to direct database queries with circuit breaker changed to returns stale cached data indefinitely and never queries database",
    },
    {
        "id": "auth",
        "original": "The authentication service issues JWTs with a 30-minute expiration signed using RS256. Refresh tokens are stored in an HttpOnly cookie and rotated on each use. The token payload includes user_id, role, and org_id claims. Passwords are hashed with bcrypt using a cost factor of 12 and a per-user salt generated by the OS random number generator.",
        "minor_change": "The authentication service issues JWTs with a 30-minute expiration signed using RS256. Refresh tokens are stored in an HttpOnly cookie and rotated on each use. The token payload includes user_id, role, and org_id claims. Passwords are hashed with bcrypt using a cost factor of 8 and a per-user salt generated by the OS random number generator.",
        "minor_detail": "cost factor of 12 changed to cost factor of 8",
        "major_change": "The authentication service issues JWTs with a 30-minute expiration signed using RS256. Refresh tokens are stored in an HttpOnly cookie and rotated on each use. The token payload includes user_id, role, and org_id claims. Passwords are stored in plaintext in the database because hashing was deemed unnecessary for internal services.",
        "major_detail": "hashed with bcrypt using cost factor 12 changed to stored in plaintext because hashing deemed unnecessary",
    },
]

DISRUPTORS = [
    "The team switched from Jira to Linear last quarter after a company-wide survey showed 73 percent of engineers preferred Linear's keyboard-driven interface. Migration of 4,200 existing tickets took two weeks.",
    "The office coffee machine was replaced with a pour-over station after the facilities team calculated it would save $3,400 per year in maintenance costs. Most engineers now bring their own mugs from home.",
    "The annual developer survey found that 61 percent of respondents use dark mode in their IDE, 28 percent use light mode, and the remaining 11 percent switch depending on the time of day.",
]

CONFIDENCE_SUFFIX = (
    "\n\nAfter stating the change (or NO CHANGE), rate your confidence on a scale of 1-5:\n"
    "1 = Very unsure, 2 = Somewhat unsure, 3 = Neutral, 4 = Somewhat confident, 5 = Very confident\n\n"
    "Format:\n"
    "Change: <your answer>\n"
    "Confidence: <1-5>"
)

random.seed(123)
data = []
task_id = 0

for passage in PASSAGES:
    for change_type in ["minor", "major"]:
        changed_text = passage[f"{change_type}_change"]
        expected_detail = passage[f"{change_type}_detail"]

        for disruptor_count in [0, 1, 3]:
            disruptor_text = ""
            if disruptor_count > 0:
                selected = random.sample(DISRUPTORS, min(disruptor_count, len(DISRUPTORS)))
                if disruptor_count > len(DISRUPTORS):
                    selected = selected * (disruptor_count // len(DISRUPTORS) + 1)
                    selected = selected[:disruptor_count]
                disruptor_text = "\n\n".join(selected)

            if disruptor_count == 0:
                prompt = (
                    "Read Version A and Version B of the following passage carefully. "
                    "Identify what factual detail changed between them.\n\n"
                    f"=== VERSION A ===\n{passage['original']}\n\n"
                    f"=== VERSION B ===\n{changed_text}\n\n"
                    "What specific factual detail changed between Version A and Version B? "
                    "State the change precisely: what was the original detail and what did it become?"
                    + CONFIDENCE_SUFFIX
                )
            else:
                prompt = (
                    "Read Version A, then the interlude, then Version B carefully. "
                    "Identify what factual detail changed between the two versions.\n\n"
                    f"=== VERSION A ===\n{passage['original']}\n\n"
                    f"=== INTERLUDE ===\n{disruptor_text}\n\n"
                    f"=== VERSION B ===\n{changed_text}\n\n"
                    "What specific factual detail changed between Version A and Version B? "
                    "Ignore the interlude — it is unrelated. "
                    "State the change precisely: what was the original detail and what did it become?"
                    + CONFIDENCE_SUFFIX
                )

            data.append({
                "task_id": task_id,
                "prompt": prompt,
                "expected": expected_detail,
                "passage_id": passage["id"],
                "change_type": change_type,
                "disruptor_count": disruptor_count,
            })
            task_id += 1

    # Control: no-change condition (false alarm test)
    for disruptor_count in [0, 1, 3]:
        disruptor_text = ""
        if disruptor_count > 0:
            selected = random.sample(DISRUPTORS, min(disruptor_count, len(DISRUPTORS)))
            if disruptor_count > len(DISRUPTORS):
                selected = selected * (disruptor_count // len(DISRUPTORS) + 1)
                selected = selected[:disruptor_count]
            disruptor_text = "\n\n".join(selected)

        if disruptor_count == 0:
            prompt = (
                "Read Version A and Version B of the following passage carefully. "
                "Identify what factual detail changed between them. "
                "If nothing changed, respond with exactly: NO CHANGE\n\n"
                f"=== VERSION A ===\n{passage['original']}\n\n"
                f"=== VERSION B ===\n{passage['original']}\n\n"
                "What specific factual detail changed between Version A and Version B? "
                "If the passages are identical, respond with exactly: NO CHANGE"
                + CONFIDENCE_SUFFIX
            )
        else:
            prompt = (
                "Read Version A, then the interlude, then Version B carefully. "
                "Identify what factual detail changed between the two versions. "
                "If nothing changed, respond with exactly: NO CHANGE\n\n"
                f"=== VERSION A ===\n{passage['original']}\n\n"
                f"=== INTERLUDE ===\n{disruptor_text}\n\n"
                f"=== VERSION B ===\n{passage['original']}\n\n"
                "What specific factual detail changed between Version A and Version B? "
                "Ignore the interlude — it is unrelated. "
                "If the passages are identical, respond with exactly: NO CHANGE"
                + CONFIDENCE_SUFFIX
            )

        data.append({
            "task_id": task_id,
            "prompt": prompt,
            "expected": "NO CHANGE",
            "passage_id": passage["id"],
            "change_type": "none",
            "disruptor_count": disruptor_count,
        })
        task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By change type: {dict(df_all['change_type'].value_counts())}")
print(f"By disruptor count: {dict(df_all['disruptor_count'].value_counts())}")

In [ ]:
# --- Dataset overview ---
print("=== Passages ===\n")
for p in PASSAGES:
    print(f"[{p['id']}]")
    print(f"  Original:     {p['original'][:80]}...")
    print(f"  Minor change: {p['minor_detail']}")
    print(f"  Major change: {p['major_detail']}")
    print()

print("=== Disruptors ===\n")
for i, d in enumerate(DISRUPTORS):
    print(f"  {i+1}. {d[:80]}...")
print()

print("=== Design ===")
print(f"  {len(PASSAGES)} passages × 3 change types (minor, major, none) × 3 disruptor levels (0, 1, 3) = {len(df_all)} items")

## Task Definition

In [ ]:
@kbench.task(
    name="change_blindness",
    description="Detect subtle factual changes between two passage versions separated by a disruptor paragraph"
)
def change_blindness(
    llm,
    prompt: str,
    expected: str,
    change_type: str,
    disruptor_count: int,
) -> bool:
    response = llm.prompt(prompt)
    response = strip_thinking(response)
    resp_norm = normalize(response)

    if expected == "NO CHANGE":
        return "no change" in resp_norm

    # Extract key values from expected (e.g., "500 requests per minute changed to 200 requests per minute")
    # Split on common transition phrases
    parts = re.split(r"changed to|changed from|→|->|replaced with|replaced by", expected.lower())
    if len(parts) >= 2:
        old_part = normalize(parts[0])
        new_part = normalize(parts[1])
        # Extract just the numbers/key terms from each part
        old_numbers = re.findall(r'\d[\d,]*', old_part)
        new_numbers = re.findall(r'\d[\d,]*', new_part)
        # Check if both key numbers appear in response
        if old_numbers and new_numbers:
            old_found = any(n.replace(',', '') in resp_norm.replace(',', '') for n in old_numbers)
            new_found = any(n.replace(',', '') in resp_norm.replace(',', '') for n in new_numbers)
            if old_found and new_found:
                return True
        # Fall back to checking key phrases (3+ word chunks)
        old_words = [w for w in old_part.split() if len(w) > 2]
        new_words = [w for w in new_part.split() if len(w) > 2]
        # At least half the meaningful words from each side should appear
        if old_words and new_words:
            old_match = sum(1 for w in old_words if w in resp_norm) / len(old_words)
            new_match = sum(1 for w in new_words if w in resp_norm) / len(new_words)
            return old_match >= 0.5 and new_match >= 0.5

    # Last resort: full substring
    return normalize(expected) in resp_norm

## Run Evaluation

In [ ]:
eval_df = df_all[["prompt", "expected", "change_type", "disruptor_count"]].copy()

print(f"Running {len(eval_df)} items with kbench.llm...")
runs = change_blindness.evaluate(
    llm=[kbench.llm],
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=120,
    max_attempts=2,
)
results = runs.as_dataframe()
results = results.merge(
    df_all[["prompt", "passage_id", "change_type", "disruptor_count"]],
    on="prompt", how="left", suffixes=("", "_meta")
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
# Convert bool result to float
results["correct"] = results["result"].apply(lambda r: float(r) if isinstance(r, bool) else float(r))

print("=== Detection Rate by Change Type × Disruptor Count ===")
pivot = results.groupby(["change_type", "disruptor_count"])["correct"].mean().unstack()
print(pivot.to_string(float_format="%.2f"))

print("\n=== Overall by Change Type ===")
print(results.groupby("change_type")["correct"].agg(["mean", "count"]).to_string(float_format="%.2f"))

# False alarm rate
no_change = results[results["change_type"] == "none"]
false_alarm = 1 - no_change["correct"].mean() if len(no_change) > 0 else 0
print(f"\nFalse alarm rate (saying something changed when nothing did): {false_alarm:.2%}")

# Change detection drop with disruptors
for ct in ["minor", "major"]:
    ct_data = results[results["change_type"] == ct]
    baseline = ct_data[ct_data["disruptor_count"] == 0]["correct"].mean()
    with_3 = ct_data[ct_data["disruptor_count"] == 3]["correct"].mean()
    print(f"{ct} change: baseline={baseline:.2%}, with 3 disruptors={with_3:.2%}, drop={baseline-with_3:.2%}")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for ct, marker, color in [("minor", "s", "orange"), ("major", "^", "blue"), ("none", "o", "red")]:
    subset = results[results["change_type"] == ct]
    curve = subset.groupby("disruptor_count")["correct"].mean()
    label = f"{ct} change" if ct != "none" else "no change (1 - false alarm)"
    vals = curve.values if ct != "none" else 1 - curve.values
    ax.plot(curve.index, vals, f"{marker}-", label=label, linewidth=2, markersize=8, color=color)

ax.set_xlabel("Number of Disruptor Paragraphs")
ax.set_ylabel("Detection Rate")
ax.set_title("Change Blindness: Detection Rate by Disruption Level")
ax.set_xticks([0, 1, 3])
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("change_blindness_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to change_blindness_results.png")

## Confidence Calibration Analysis

Does the model know when it's wrong? We parse the confidence ratings (1-5) from responses and compare against actual accuracy to measure metacognitive calibration.

In [ ]:
# Parse confidence from model responses
def extract_confidence(response_text: str) -> int | None:
    """Extract confidence rating (1-5) from response."""
    text = strip_thinking(str(response_text))
    # Look for "Confidence: N" pattern
    match = re.search(r"[Cc]onfidence\s*[:=]\s*(\d)", text)
    if match:
        val = int(match.group(1))
        return val if 1 <= val <= 5 else None
    return None

# Extract confidence from raw responses
if "response" in results.columns:
    results["confidence"] = results["response"].apply(extract_confidence)
else:
    # Try to get from the evaluation run
    results["confidence"] = None
    print("Note: Raw responses not available in results DataFrame.")
    print("Confidence analysis requires the raw response text.")

conf_results = results.dropna(subset=["confidence"]).copy()
conf_results["confidence"] = conf_results["confidence"].astype(int)

if len(conf_results) > 0:
    print(f"=== Confidence Calibration ({len(conf_results)} items with confidence) ===\n")
    
    # Calibration table: for each confidence level, what's the actual accuracy?
    print("Confidence | Accuracy | Count | Calibration Gap")
    print("-" * 55)
    
    calibration_data = []
    for conf_level in range(1, 6):
        subset = conf_results[conf_results["confidence"] == conf_level]
        if len(subset) > 0:
            acc = subset["correct"].mean()
            expected_acc = conf_level / 5.0  # Linear mapping: 1→0.2, 5→1.0
            gap = acc - expected_acc
            calibration_data.append({
                "confidence": conf_level,
                "accuracy": acc,
                "expected": expected_acc,
                "count": len(subset),
                "gap": gap,
            })
            print(f"    {conf_level}      |  {acc:.2%}  |  {len(subset):3d}  | {gap:+.2%}")
    
    cal_df = pd.DataFrame(calibration_data)
    
    # Expected Calibration Error (ECE)
    if len(cal_df) > 0:
        total = cal_df["count"].sum()
        ece = sum(row["count"] / total * abs(row["gap"]) for _, row in cal_df.iterrows())
        print(f"\nExpected Calibration Error (ECE): {ece:.4f}")
    
    # Overconfidence: wrong answers with confidence 4-5
    wrong_high_conf = conf_results[(conf_results["correct"] == 0) & (conf_results["confidence"] >= 4)]
    total_wrong = conf_results[conf_results["correct"] == 0]
    if len(total_wrong) > 0:
        overconf_rate = len(wrong_high_conf) / len(total_wrong)
        print(f"Overconfidence rate (wrong + conf>=4): {overconf_rate:.2%} ({len(wrong_high_conf)}/{len(total_wrong)})")
    
    # Underconfidence: correct answers with confidence 1-2
    correct_low_conf = conf_results[(conf_results["correct"] == 1) & (conf_results["confidence"] <= 2)]
    total_correct = conf_results[conf_results["correct"] == 1]
    if len(total_correct) > 0:
        underconf_rate = len(correct_low_conf) / len(total_correct)
        print(f"Underconfidence rate (correct + conf<=2): {underconf_rate:.2%} ({len(correct_low_conf)}/{len(total_correct)})")
    
    # Calibration by change type
    print("\n=== Confidence by Change Type ===")
    print(conf_results.groupby("change_type")[["confidence", "correct"]].mean().to_string(float_format="%.2f"))
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Reliability diagram
    if len(cal_df) > 0:
        ax1.bar(cal_df["confidence"], cal_df["accuracy"], width=0.6, color="steelblue", 
                edgecolor="black", linewidth=0.5, label="Actual accuracy")
        ax1.plot([1, 5], [0.2, 1.0], "r--", linewidth=2, label="Perfect calibration")
        ax1.set_xlabel("Confidence Level")
        ax1.set_ylabel("Actual Accuracy")
        ax1.set_title("Reliability Diagram")
        ax1.set_xticks([1, 2, 3, 4, 5])
        ax1.set_ylim(0, 1.1)
        ax1.legend()
    
    # Confidence distribution for correct vs incorrect
    for label, subset, color in [
        ("Correct", conf_results[conf_results["correct"] == 1], "seagreen"),
        ("Incorrect", conf_results[conf_results["correct"] == 0], "tomato"),
    ]:
        if len(subset) > 0:
            counts = subset["confidence"].value_counts().reindex(range(1, 6), fill_value=0)
            offset = 0.2 if label == "Correct" else -0.2
            ax2.bar(counts.index + offset, counts.values, width=0.35, color=color,
                    edgecolor="black", linewidth=0.5, label=label)
    
    ax2.set_xlabel("Confidence Level")
    ax2.set_ylabel("Count")
    ax2.set_title("Confidence Distribution: Correct vs Incorrect")
    ax2.set_xticks([1, 2, 3, 4, 5])
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig("change_blindness_calibration.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Calibration plot saved to change_blindness_calibration.png")
else:
    print("No confidence data available — model may not have followed the confidence format.")